Smart Phone Search — Vector Embedding với ChromaDB

**Mục tiêu:** Nhúng toàn bộ 915 điện thoại vào Vector DB để tìm kiếm bằng ngôn ngữ tự nhiên.

**Stack:**
- `sentence-transformers` (multilingual, offline, miễn phí)
- `ChromaDB` (Vector DB local)
- Fix lỗi PyTorch cũ (>= 2.2)

Bước 0: Fix môi trường — Cài đúng phiên bản

In [1]:
#  Fix lỗi: PyTorch 2.2 không tương thích sentence-transformers mới
# Chạy cell này 1 lần duy nhất, sau đó RESTART KERNEL

import subprocess, sys

packages = [
    # Downgrade torch về bản tương thích
    "torch==2.2.0",
    # sentence-transformers bản cũ chạy được với torch 2.2
    "sentence-transformers==2.7.0",
    # transformers phiên bản ổn định
    "transformers==4.40.2",
    # ChromaDB
    "chromadb",
    # Các thư viện hỗ trợ
    "pandas",
    "numpy",
    "tqdm",
]

print(" Đang cài đặt packages...")
for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    status = "✅" if result.returncode == 0 else "❌"
    print(f"  {status} {pkg}")

print("\n XONG! Hãy RESTART KERNEL trước khi chạy tiếp (Menu: Kernel > Restart)")

 Đang cài đặt packages...
  ✅ torch==2.2.0
  ✅ sentence-transformers==2.7.0
  ✅ transformers==4.40.2
  ✅ chromadb
  ✅ pandas
  ✅ numpy
  ✅ tqdm

 XONG! Hãy RESTART KERNEL trước khi chạy tiếp (Menu: Kernel > Restart)


Bước 1: Import thư viện & Kiểm tra môi trường

In [2]:
import pandas as pd
import numpy as np
import chromadb
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import os
import json
import warnings
warnings.filterwarnings('ignore')

print("=" * 50)
print(" Kiểm tra môi trường")
print("=" * 50)
print(f"  Python      : {sys.version.split()[0]}")
print(f"  PyTorch     : {torch.__version__}")

import sentence_transformers
print(f"  ST version  : {sentence_transformers.__version__}")
print(f"  ChromaDB    : {chromadb.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()} (dùng CPU nếu False)")
print("=" * 50)
print(" Môi trường sẵn sàng!")

/Users/admin/smart-phone-search/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Kiểm tra môi trường
  Python      : 3.11.9
  PyTorch     : 2.2.0
  ST version  : 2.7.0
  ChromaDB    : 1.5.9
  CUDA available: False (dùng CPU nếu False)
 Môi trường sẵn sàng!


Bước 2: Load và xem dữ liệu

In [3]:
# Load dữ liệu sạch
DATA_PATH = "../data/phones_clean.csv"  # Chỉnh đường dẫn nếu cần

df = pd.read_csv(DATA_PATH)

print(f" Đã load thành công: {len(df)} điện thoại")
print(f" Cột dữ liệu: {df.columns.tolist()}")
print()
df.head(3)

 Đã load thành công: 915 điện thoại
 Cột dữ liệu: ['Brand', 'Model', 'Launched Year', 'Processor', 'RAM_GB', 'RAM_Variants', 'Screen_Size_Inch', 'Weight_g', 'Battery_mAh', 'Front_Camera_MP', 'Back_Camera_Primary_MP', 'Back_Camera_Lens_Count', 'Back_Camera_Raw', 'Price_Pakistan', 'Price_India', 'Price_China', 'Price_USA', 'Price_Dubai']



,Brand,Model,Launched Year,Processor,RAM_GB,RAM_Variants,Screen_Size_Inch,Weight_g,Battery_mAh,Front_Camera_MP,Back_Camera_Primary_MP,Back_Camera_Lens_Count,Back_Camera_Raw,Price_Pakistan,Price_India,Price_China,Price_USA,Price_Dubai
0,Apple,iPhone 16 128GB,2024.0,A17 Bionic,6.0,6GB,6.1,174.0,3600.0,12.0,48.0,1.0,48MP,224999.0,79999.0,5799.0,799.0,2799.0
1,Apple,iPhone 16 256GB,2024.0,A17 Bionic,6.0,6GB,6.1,174.0,3600.0,12.0,48.0,1.0,48MP,234999.0,84999.0,6099.0,849.0,2999.0
2,Apple,iPhone 16 512GB,2024.0,A17 Bionic,6.0,6GB,6.1,174.0,3600.0,12.0,48.0,1.0,48MP,244999.0,89999.0,6499.0,899.0,3199.0


In [4]:
# Thống kê nhanh
print(" Phân phối thương hiệu:")
print(df['Brand'].value_counts().head(10).to_string())
print()
print(f" Năm ra mắt: {df['Launched Year'].min():.0f} → {df['Launched Year'].max():.0f}")
print(f" Giá USD  : ${df['Price_USA'].min():.0f} → ${df['Price_USA'].max():.0f}")
print(f" Pin      : {df['Battery_mAh'].min():.0f} → {df['Battery_mAh'].max():.0f} mAh")

 Phân phối thương hiệu:
Brand
Oppo        115
Apple        97
Honor        91
Samsung      88
Vivo         86
Realme       69
Motorola     62
Infinix      55
OnePlus      53
Huawei       40

 Năm ra mắt: 2014 → 2025
 Giá USD  : $79 → $39622
 Pin      : 2000 → 11200 mAh


Bước 3: Tạo văn bản mô tả (Document) cho mỗi điện thoại

In [5]:
def create_description(row):
    """
    Ghép các thông số thành 1 đoạn văn mô tả đầy đủ.
    Đây là văn bản sẽ được embedding thành vector.
    Viết cả tiếng Anh để model multilingual hiểu tốt hơn.
    """
    # Xử lý giá trị null
    brand        = str(row['Brand'])                    if pd.notna(row['Brand'])                    else 'Unknown'
    model        = str(row['Model'])                    if pd.notna(row['Model'])                    else 'Unknown'
    year         = int(row['Launched Year'])            if pd.notna(row['Launched Year'])            else 0
    processor    = str(row['Processor'])                if pd.notna(row['Processor'])                else 'Unknown'
    ram          = float(row['RAM_GB'])                 if pd.notna(row['RAM_GB'])                   else 0
    screen       = float(row['Screen_Size_Inch'])       if pd.notna(row['Screen_Size_Inch'])         else 0
    weight       = float(row['Weight_g'])               if pd.notna(row['Weight_g'])                 else 0
    battery      = float(row['Battery_mAh'])            if pd.notna(row['Battery_mAh'])              else 0
    front_cam    = float(row['Front_Camera_MP'])        if pd.notna(row['Front_Camera_MP'])          else 0
    back_cam     = float(row['Back_Camera_Primary_MP']) if pd.notna(row['Back_Camera_Primary_MP'])   else 0
    lens_count   = int(row['Back_Camera_Lens_Count'])   if pd.notna(row['Back_Camera_Lens_Count'])   else 0
    price_usd    = float(row['Price_USA'])              if pd.notna(row['Price_USA'])                else 0

    # Phân loại pin
    battery_label = "large battery" if battery >= 5000 else ("medium battery" if battery >= 4000 else "compact battery")

    # Phân loại giá
    if price_usd < 300:
        price_label = "budget phone giá rẻ"
    elif price_usd < 600:
        price_label = "mid-range tầm trung"
    elif price_usd < 1000:
        price_label = "high-end cao cấp"
    else:
        price_label = "flagship premium đỉnh cao"

    # Phân loại RAM
    ram_label = "gaming RAM cao" if ram >= 12 else ("đa nhiệm tốt" if ram >= 8 else "cơ bản")

    desc = (
        f"{brand} {model} launched in {year}. "
        f"Powered by {processor} processor with {ram}GB RAM ({ram_label}). "
        f"Screen size {screen} inch display. "
        f"Weight {weight}g. "
        f"Battery {battery}mAh ({battery_label}). "
        f"Front camera {front_cam}MP, rear camera {back_cam}MP with {lens_count} lenses. "
        f"Price ${price_usd} USD — {price_label}."
    )
    return desc

# Áp dụng cho toàn bộ dataset
df['description'] = df.apply(create_description, axis=1)

print(" Đã tạo description cho tất cả điện thoại")
print()
print(" Ví dụ description đầu tiên:")
print("-" * 60)
print(df['description'].iloc[0])
print("-" * 60)
print(f"\n Ví dụ điện thoại giá rẻ:")
cheap = df[df['Price_USA'] < 200].iloc[0]
print(cheap['description'])

 Đã tạo description cho tất cả điện thoại

 Ví dụ description đầu tiên:
------------------------------------------------------------
Apple iPhone 16 128GB launched in 2024. Powered by A17 Bionic processor with 6.0GB RAM (cơ bản). Screen size 6.1 inch display. Weight 174.0g. Battery 3600.0mAh (compact battery). Front camera 12.0MP, rear camera 48.0MP with 1 lenses. Price $799.0 USD — high-end cao cấp.
------------------------------------------------------------

 Ví dụ điện thoại giá rẻ:
Samsung Galaxy A04 64GB launched in 2022. Powered by Exynos 850 processor with 4.0GB RAM (cơ bản). Screen size 6.5 inch display. Weight 188.0g. Battery 5000.0mAh (large battery). Front camera 5.0MP, rear camera 50.0MP with 2 lenses. Price $199.0 USD — budget phone giá rẻ.


Bước 4: Load Embedding Model

In [6]:
# Model multilingual — hiểu cả tiếng Việt lẫn tiếng Anh
# paraphrase-multilingual-MiniLM-L12-v2:
#   - Nhẹ (~420MB), nhanh
#   - Hỗ trợ 50+ ngôn ngữ bao gồm tiếng Việt
#   - Vector 384 chiều

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

print(f"⏳ Đang tải model: {MODEL_NAME}")
print("   (Lần đầu sẽ download ~420MB, các lần sau load từ cache)")

embedding_model = SentenceTransformer(MODEL_NAME)

# Test nhanh
test_vec = embedding_model.encode(["test"])
print(f"\n Model đã sẵn sàng!")
print(f"   Vector dimension: {test_vec.shape[1]} chiều")
print(f"   Device: {embedding_model.device}")

⏳ Đang tải model: paraphrase-multilingual-MiniLM-L12-v2
   (Lần đầu sẽ download ~420MB, các lần sau load từ cache)

 Model đã sẵn sàng!
   Vector dimension: 384 chiều
   Device: mps:0


Bước 5: Tạo Embeddings cho toàn bộ 915 điện thoại

In [7]:
import time

descriptions = df['description'].tolist()

print(f" Bắt đầu embedding {len(descriptions)} điện thoại...")
print("   (Khoảng 1-3 phút tùy CPU)")
print()

start = time.time()

embeddings = embedding_model.encode(
    descriptions,
    batch_size=64,          # Xử lý 64 cái 1 lúc
    show_progress_bar=True, # Hiển thị thanh tiến trình
    convert_to_numpy=True,  # Output dạng numpy array
    normalize_embeddings=True  # Chuẩn hoá để cosine similarity chính xác hơn
)

elapsed = time.time() - start

print(f"\n Embedding hoàn tất!")
print(f"   Thời gian  : {elapsed:.1f} giây")
print(f"   Shape      : {embeddings.shape}  ({embeddings.shape[0]} phones × {embeddings.shape[1]} dimensions)")
print(f"   Kích thước : {embeddings.nbytes / 1024 / 1024:.1f} MB trong bộ nhớ")

 Bắt đầu embedding 915 điện thoại...
   (Khoảng 1-3 phút tùy CPU)



Batches: 100%|██████████| 15/15 [01:38<00:00,  6.54s/it]


 Embedding hoàn tất!
   Thời gian  : 98.1 giây
   Shape      : (915, 384)  (915 phones × 384 dimensions)
   Kích thước : 1.3 MB trong bộ nhớ


Bước 6: Lưu vào ChromaDB

In [8]:
# Đường dẫn lưu ChromaDB (persistent = lưu vĩnh viễn trên ổ cứng)
CHROMA_PATH = "../data/chroma_db"
COLLECTION_NAME = "phones"

os.makedirs(CHROMA_PATH, exist_ok=True)

# Khởi tạo ChromaDB
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Xoá collection cũ nếu đã tồn tại (để chạy lại từ đầu)
try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print(f"  Đã xoá collection cũ: '{COLLECTION_NAME}'")
except:
    pass

# Tạo collection mới
collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}  # Dùng cosine similarity
)

print(f" Tạo collection mới: '{COLLECTION_NAME}'")

  Đã xoá collection cũ: 'phones'
 Tạo collection mới: 'phones'


In [9]:
# Chuẩn bị dữ liệu để insert
ids        = [str(i) for i in df.index]
documents  = df['description'].tolist()
embed_list = embeddings.tolist()

# Metadata: các trường dùng để filter sau này
metadatas = []
for _, row in df.iterrows():
    metadatas.append({
        "brand"     : str(row['Brand'])                    if pd.notna(row['Brand'])          else "",
        "model"     : str(row['Model'])                    if pd.notna(row['Model'])          else "",
        "year"      : int(row['Launched Year'])            if pd.notna(row['Launched Year'])  else 0,
        "ram_gb"    : float(row['RAM_GB'])                 if pd.notna(row['RAM_GB'])         else 0.0,
        "battery"   : float(row['Battery_mAh'])            if pd.notna(row['Battery_mAh'])    else 0.0,
        "price_usd" : float(row['Price_USA'])              if pd.notna(row['Price_USA'])      else 0.0,
        "screen"    : float(row['Screen_Size_Inch'])       if pd.notna(row['Screen_Size_Inch']) else 0.0,
        "front_cam" : float(row['Front_Camera_MP'])        if pd.notna(row['Front_Camera_MP']) else 0.0,
        "back_cam"  : float(row['Back_Camera_Primary_MP']) if pd.notna(row['Back_Camera_Primary_MP']) else 0.0,
        "processor" : str(row['Processor'])                if pd.notna(row['Processor'])      else "",
    })

# Insert vào ChromaDB theo batch 200
BATCH_SIZE = 200
total = len(ids)

print(f" Đang lưu {total} điện thoại vào ChromaDB...")

for start_idx in range(0, total, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, total)
    collection.add(
        ids        = ids[start_idx:end_idx],
        embeddings = embed_list[start_idx:end_idx],
        documents  = documents[start_idx:end_idx],
        metadatas  = metadatas[start_idx:end_idx],
    )
    print(f"    Đã lưu {end_idx}/{total}")

print(f"\n Hoàn tất! ChromaDB có {collection.count()} điện thoại")
print(f" Dữ liệu lưu tại: {os.path.abspath(CHROMA_PATH)}")

 Đang lưu 915 điện thoại vào ChromaDB...
    Đã lưu 200/915
    Đã lưu 400/915
    Đã lưu 600/915
    Đã lưu 800/915
    Đã lưu 915/915

 Hoàn tất! ChromaDB có 915 điện thoại
 Dữ liệu lưu tại: /Users/admin/smart-phone-search/data/chroma_db


## Bước 7: Test tìm kiếm bằng ngôn ngữ tự nhiên

In [10]:
def search_phones(query: str, n_results: int = 5, filters: dict = None):
    """
    Tìm kiếm điện thoại bằng ngôn ngữ tự nhiên.
    
    Args:
        query     : Câu hỏi/yêu cầu của người dùng
        n_results : Số kết quả trả về
        filters   : dict metadata để filter (vd: {"brand": "Samsung"})
    
    Returns:
        DataFrame kết quả tìm kiếm
    """
    # Embed query
    query_vector = embedding_model.encode([query], normalize_embeddings=True).tolist()
    
    # Build where clause cho filter
    where_clause = None
    if filters:
        conditions = []
        for key, val in filters.items():
            if isinstance(val, str):
                conditions.append({key: {"$eq": val}})
            elif isinstance(val, dict):  # range filter
                conditions.append({key: val})
        where_clause = {"$and": conditions} if len(conditions) > 1 else conditions[0]
    
    # Query ChromaDB
    results = collection.query(
        query_embeddings = query_vector,
        n_results        = n_results,
        where            = where_clause,
        include          = ["documents", "metadatas", "distances"]
    )
    
    # Format kết quả
    rows = []
    for i in range(len(results['ids'][0])):
        meta  = results['metadatas'][0][i]
        score = 1 - results['distances'][0][i]  # cosine similarity (0-1)
        rows.append({
            'Rank'      : i + 1,
            'Score'     : f"{score:.3f}",
            'Brand'     : meta['brand'],
            'Model'     : meta['model'],
            'Year'      : meta['year'],
            'RAM (GB)'  : meta['ram_gb'],
            'Battery'   : f"{meta['battery']:.0f} mAh",
            'Price USD' : f"${meta['price_usd']:.0f}",
            'Processor' : meta['processor'],
        })
    
    result_df = pd.DataFrame(rows)
    return result_df


def print_search(query, **kwargs):
    print(f"\n{'='*60}")
    print(f" Query: '{query}'")
    if kwargs.get('filters'):
        print(f"   Filter: {kwargs['filters']}")
    print(f"{'='*60}")
    result = search_phones(query, **kwargs)
    print(result.to_string(index=False))
    return result


print(" Hàm search_phones đã sẵn sàng!")

 Hàm search_phones đã sẵn sàng!


In [11]:
# ===== TEST 1: Tìm kiếm tiếng Việt =====
print_search("điện thoại pin trâu nhất hiện nay")


 Query: 'điện thoại pin trâu nhất hiện nay'
 Rank Score   Brand                      Model  Year  RAM (GB)  Battery Price USD          Processor
    1 0.431   Nokia                        T21  2022       4.0 8200 mAh    $39622        Unisoc T612
    2 0.411 Samsung Galaxy Note 20 Ultra 128GB  2020      12.0 4500 mAh     $1299         Exynos 990
    3 0.410 Samsung Galaxy Note 20 Ultra 256GB  2020      12.0 4500 mAh     $1399         Exynos 990
    4 0.408 Samsung      Galaxy Z Flip 5 256GB  2023       8.0 3700 mAh      $999 Snapdragon 8 Gen 2
    5 0.407 Samsung       Galaxy Note 20 256GB  2020       8.0 4300 mAh     $1099         Exynos 990


,Rank,Score,Brand,Model,Year,RAM (GB),Battery,Price USD,Processor
0,1,0.431,Nokia,T21,2022,4.0,8200 mAh,$39622,Unisoc T612
1,2,0.411,Samsung,Galaxy Note 20 Ultra 128GB,2020,12.0,4500 mAh,$1299,Exynos 990
2,3,0.410,Samsung,Galaxy Note 20 Ultra 256GB,2020,12.0,4500 mAh,$1399,Exynos 990
3,4,0.408,Samsung,Galaxy Z Flip 5 256GB,2023,8.0,3700 mAh,$999,Snapdragon 8 Gen 2
4,5,0.407,Samsung,Galaxy Note 20 256GB,2020,8.0,4300 mAh,$1099,Exynos 990


In [12]:
# ===== TEST 2: Tìm theo tiếng Anh =====
print_search("best camera phone flagship 2024")


 Query: 'best camera phone flagship 2024'
 Rank Score Brand               Model  Year  RAM (GB)  Battery Price USD                Processor
    1 0.522  Oppo         K9 5G 256GB  2021       8.0 4300 mAh      $299 Qualcomm Snapdragon 768G
    2 0.517  Oppo Reno6 Pro+ 5G 256GB  2021      12.0 4500 mAh      $599           Snapdragon 870
    3 0.509  Oppo     K9 Pro 5G 256GB  2021      12.0 4500 mAh      $329  MediaTek Dimensity 1200
    4 0.508  Oppo         K7 5G 256GB  2020       8.0 4025 mAh      $299 Qualcomm Snapdragon 765G
    5 0.506  Oppo     K9 Pro 5G 128GB  2021       8.0 4500 mAh      $299  MediaTek Dimensity 1200


,Rank,Score,Brand,Model,Year,RAM (GB),Battery,Price USD,Processor
0,1,0.522,Oppo,K9 5G 256GB,2021,8.0,4300 mAh,$299,Qualcomm Snapdragon 768G
1,2,0.517,Oppo,Reno6 Pro+ 5G 256GB,2021,12.0,4500 mAh,$599,Snapdragon 870
2,3,0.509,Oppo,K9 Pro 5G 256GB,2021,12.0,4500 mAh,$329,MediaTek Dimensity 1200
3,4,0.508,Oppo,K7 5G 256GB,2020,8.0,4025 mAh,$299,Qualcomm Snapdragon 765G
4,5,0.506,Oppo,K9 Pro 5G 128GB,2021,8.0,4500 mAh,$299,MediaTek Dimensity 1200


In [13]:
# ===== TEST 3: Tìm điện thoại giá rẻ =====
print_search("điện thoại giá rẻ dưới 300 đô cho sinh viên")


 Query: 'điện thoại giá rẻ dưới 300 đô cho sinh viên'


 Rank Score    Brand                  Model  Year  RAM (GB)  Battery Price USD          Processor
    1 0.373    Nokia               C32 64GB  2023       3.0 5000 mAh      $139     Unisoc SC9863A
    2 0.365    Nokia              C32 128GB  2023       4.0 5000 mAh      $159     Unisoc SC9863A
    3 0.363 Motorola      Edge 30 Neo 256GB  2022       8.0 4020 mAh      $549  Snapdragon 695 5G
    4 0.361 Motorola      Edge 30 Neo 128GB  2022       6.0 4020 mAh      $499  Snapdragon 695 5G
    5 0.361  Samsung Galaxy S23 Ultra 256GB  2023      12.0 5000 mAh     $1299 Snapdragon 8 Gen 2


,Rank,Score,Brand,Model,Year,RAM (GB),Battery,Price USD,Processor
0,1,0.373,Nokia,C32 64GB,2023,3.0,5000 mAh,$139,Unisoc SC9863A
1,2,0.365,Nokia,C32 128GB,2023,4.0,5000 mAh,$159,Unisoc SC9863A
2,3,0.363,Motorola,Edge 30 Neo 256GB,2022,8.0,4020 mAh,$549,Snapdragon 695 5G
3,4,0.361,Motorola,Edge 30 Neo 128GB,2022,6.0,4020 mAh,$499,Snapdragon 695 5G
4,5,0.361,Samsung,Galaxy S23 Ultra 256GB,2023,12.0,5000 mAh,$1299,Snapdragon 8 Gen 2


In [14]:
# ===== TEST 4: Kết hợp Vector Search + Filter Metadata =====
print_search(
    "gaming phone RAM cao màn hình lớn",
    filters={"brand": "Samsung"},
    n_results=5
)


 Query: 'gaming phone RAM cao màn hình lớn'
   Filter: {'brand': 'Samsung'}
 Rank Score   Brand                     Model  Year  RAM (GB)   Battery Price USD          Processor
    1 0.559 Samsung Galaxy Tab S9 Ultra 256GB  2023      12.0 11200 mAh     $1199 Snapdragon 8 Gen 2
    2 0.559 Samsung        Galaxy C9 Pro 64GB  2016       6.0  4000 mAh      $399     Snapdragon 653
    3 0.557 Samsung      Galaxy Tab S9+ 256GB  2023      12.0 10090 mAh      $999 Snapdragon 8 Gen 2
    4 0.557 Samsung Galaxy Tab S8 Ultra 256GB  2022      12.0 11200 mAh     $1099 Snapdragon 8 Gen 1
    5 0.553 Samsung    Galaxy Tab E 10.1 16GB  2016       1.5  5000 mAh      $129  Spreadtrum SC8830


,Rank,Score,Brand,Model,Year,RAM (GB),Battery,Price USD,Processor
0,1,0.559,Samsung,Galaxy Tab S9 Ultra 256GB,2023,12.0,11200 mAh,$1199,Snapdragon 8 Gen 2
1,2,0.559,Samsung,Galaxy C9 Pro 64GB,2016,6.0,4000 mAh,$399,Snapdragon 653
2,3,0.557,Samsung,Galaxy Tab S9+ 256GB,2023,12.0,10090 mAh,$999,Snapdragon 8 Gen 2
3,4,0.557,Samsung,Galaxy Tab S8 Ultra 256GB,2022,12.0,11200 mAh,$1099,Snapdragon 8 Gen 1
4,5,0.553,Samsung,Galaxy Tab E 10.1 16GB,2016,1.5,5000 mAh,$129,Spreadtrum SC8830


In [15]:
# ===== TEST 5: Filter theo khoảng giá =====
print_search(
    "điện thoại tốt nhất camera selfie",
    filters={"price_usd": {"$lte": 500}},  # giá dưới $500
    n_results=5
)


 Query: 'điện thoại tốt nhất camera selfie'
   Filter: {'price_usd': {'$lte': 500}}
 Rank Score    Brand              Model  Year  RAM (GB)  Battery Price USD           Processor
    1 0.348 Motorola Edge 50 Lite 256GB  2024       8.0 5000 mAh      $449      Snapdragon 695
    2 0.347 Motorola One Vision 3 256GB  2024       8.0 3500 mAh      $349         Exynos 9609
    3 0.341 Motorola Edge 50 Lite 128GB  2024       6.0 5000 mAh      $399      Snapdragon 695
    4 0.337 Motorola      Edge 50 256GB  2024       8.0 4700 mAh      $450 Snapdragon 7s Gen 2
    5 0.336 Motorola One Vision 3 128GB  2024       6.0 3500 mAh      $299         Exynos 9609


,Rank,Score,Brand,Model,Year,RAM (GB),Battery,Price USD,Processor
0,1,0.348,Motorola,Edge 50 Lite 256GB,2024,8.0,5000 mAh,$449,Snapdragon 695
1,2,0.347,Motorola,One Vision 3 256GB,2024,8.0,3500 mAh,$349,Exynos 9609
2,3,0.341,Motorola,Edge 50 Lite 128GB,2024,6.0,5000 mAh,$399,Snapdragon 695
3,4,0.337,Motorola,Edge 50 256GB,2024,8.0,4700 mAh,$450,Snapdragon 7s Gen 2
4,5,0.336,Motorola,One Vision 3 128GB,2024,6.0,3500 mAh,$299,Exynos 9609


In [16]:
# ===== TEST 6: Thử nhiều query khác nhau =====
queries = [
    "điện thoại nhẹ nhất dưới 150g",
    "iPhone mới nhất 2024",
    "Android chip Snapdragon mạnh nhất",
    "điện thoại camera 200MP",
    "budget phone good battery life",
]

for q in queries:
    print_search(q, n_results=3)


 Query: 'điện thoại nhẹ nhất dưới 150g'
 Rank Score   Brand             Model  Year  RAM (GB)  Battery Price USD               Processor
    1 0.507 Infinix   Hot 30 5G 128GB  2023       8.0 5000 mAh      $239  MediaTek Dimensity 810
    2 0.505 Infinix Hot 30i NFC 128GB  2023       4.0 5000 mAh      $159             Unisoc T606
    3 0.492   Tecno Spark 30 5G 128GB  2024       6.0 5000 mAh      $229 MediaTek Dimensity 6020

 Query: 'iPhone mới nhất 2024'
 Rank Score Brand               Model  Year  RAM (GB)  Battery Price USD  Processor
    1 0.576 Apple iPhone 13 Pro 256GB  2021       6.0 3095 mAh     $1099 A15 Bionic
    2 0.568 Apple iPhone 16 Pro 256GB  2024       8.0 4400 mAh     $1049    A17 Pro
    3 0.557 Apple     iPhone 13 256GB  2021       4.0 3240 mAh      $899 A15 Bionic

 Query: 'Android chip Snapdragon mạnh nhất'
 Rank Score    Brand                Model  Year  RAM (GB)  Battery Price USD          Processor
    1 0.563 Motorola  Edge 50 Ultra 512GB  2024      12.0 5000

Bước 8: Load lại ChromaDB (cho lần sau)

In [17]:
# === Hàm tiện ích: Load ChromaDB đã lưu ===
# (Dùng trong notebook khác hoặc khi restart)

def load_phone_db(chroma_path="../data/chroma_db", collection_name="phones"):
    """
    Load ChromaDB đã được build sẵn.
    Dùng trong API server hoặc notebook khác.
    """
    client     = chromadb.PersistentClient(path=chroma_path)
    collection = client.get_collection(collection_name)
    model      = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    print(f" Loaded {collection.count()} phones from {chroma_path}")
    return collection, model

# Ví dụ dùng trong notebook khác:
# collection, embedding_model = load_phone_db()
# result = search_phones("điện thoại pin trâu")

print(" Hàm load_phone_db sẵn sàng")
print(f"\n Thống kê ChromaDB hiện tại:")
print(f"   Collection : '{COLLECTION_NAME}'")
print(f"   Số lượng   : {collection.count()} điện thoại")
print(f"   Lưu tại    : {os.path.abspath(CHROMA_PATH)}")

 Hàm load_phone_db sẵn sàng

 Thống kê ChromaDB hiện tại:
   Collection : 'phones'
   Số lượng   : 915 điện thoại
   Lưu tại    : /Users/admin/smart-phone-search/data/chroma_db


## ✅ Tóm tắt

| Bước | Mô tả | Kết quả |
|------|--------|----------|
| 1 | Fix môi trường | torch 2.2 + ST 2.7 |
| 2 | Load dữ liệu | 915 điện thoại |
| 3 | Tạo description | Văn bản mô tả đầy đủ |
| 4 | Load model | `paraphrase-multilingual-MiniLM-L12-v2` |
| 5 | Tạo embeddings | 915 × 384 vectors |
| 6 | Lưu ChromaDB | Persistent tại `../data/chroma_db` |
| 7 | Test search | Tìm kiếm tiếng Việt + filter |

**Bước tiếp theo:** Tạo `03_api_server.ipynb` với FastAPI để expose search endpoint 🚀